In [1]:
import re
import math
import requests
import pandas as pd
from dateutil import parser as dateparser
from datetime import datetime

In [ ]:
lok_sabha_url = "https://en.wikipedia.org/wiki/List_of_Indian_general_elections"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

response = requests.get(lok_sabha_url, headers=headers, timeout=30)
response.raise_for_status()
tables = pd.read_html(response.text)
# Manually identify the main elections table – usually the first with 'Election year'
elections_df = None
for t in tables:
    if "Election year" in t.columns or "Election year" in t.iloc[0].astype(str).tolist():
        elections_df = t
        break

if elections_df is None:
    raise RuntimeError("Could not find Lok Sabha elections table; inspect `tables` manually.")

# Normalize column names
elections_df.columns = [c.strip().lower().replace(" ", "_") for c in elections_df.columns]

# Keep only key columns; adjust names if they differ
keep_cols = [
    "election_year",        # may be 'Election year'
    "lok_sabha",            # 'Lok Sabha'
    "party_in_government",  # 'Party in government'
    "seats_won_by_the_ruling_party",  # might differ slightly
    "prime_minister",       # 'Prime Minister'
    "turnout"               # optional
]
keep_cols = [c for c in keep_cols if c in elections_df.columns]
elections_df = elections_df[keep_cols].copy()

# Clean up year (e.g. '1951–52' -> 1952 as end year)
def parse_year_range(s):
    s = str(s).strip()
    if "–" in s or "-" in s:
        parts = re.split(r"[–-]", s)
        parts = [p.strip() for p in parts if p.strip()]
        try:
            return int(parts[-1])
        except ValueError:
            return None
    try:
        return int(s)
    except ValueError:
        return None

elections_df["election_year_numeric"] = elections_df["election_year"].apply(parse_year_range)

# Basic cleanup of party/alliance labels
def clean_party(x):
    x = str(x).strip()
    if not x:
        return None
    return x

elections_df["party_in_government"] = elections_df["party_in_government"].apply(clean_party)

# Save
elections_df.to_csv("lok_sabha_ruling_party.csv", index=False)

HTTPError: HTTP Error 403: Forbidden

In [ ]:
def parse_year_span(span, default_end_year=None):
    """
    Parse 'YYYY - YYYY' or 'YYYY - Present' into datetime start/end.
    Uses Jan 1 for start and Dec 31 for end for simplicity.
    """
    span = str(span).strip()
    if not span or span.lower() in {"na", "n/a"}:
        return None, None

    parts = re.split(r"-|–|—", span)
    parts = [p.strip() for p in parts if p.strip()]
    if not parts:
        return None, None

    start_year = None
    end_year = None

    try:
        start_year = int(parts[0][:4])
    except ValueError:
        pass

    if len(parts) > 1:
        end_raw = parts[1].lower()
        if "present" in end_raw or "incumbent" in end_raw:
            end_year = default_end_year or datetime.now().year
        else:
            try:
                end_year = int(parts[1][:4])
            except ValueError:
                end_year = default_end_year or datetime.now().year
    else:
        end_year = default_end_year or datetime.now().year

    if start_year is None or end_year is None:
        return None, None

    start_date = datetime(start_year, 1, 1)
    end_date = datetime(end_year, 12, 31)
    return start_date, end_date